# dk EGAT 모델 풀 학습 (Colab A100)

> **최종 업데이트**: 2026-07-14: **손실함수를 BCE+threshold sweep → Adaptive Thresholding
> (ATLoss/PUATLoss)으로 교체.** 첫 실행에서 distant 프리트레인만으로 dev F1 24.77이 나왔는데,
> 같은 20k distant subset 기준 RoBERTa+LCP+ATLoss의 43.15(`EXPERIMENTS.md` 실험 2)보다도 낮아
> 원인 진단 — BCE는 DocRED 97% NA 불균형을 학습 중엔 안 다루고 사후 threshold sweep으로만
> 보정하는데, train_loss가 0.6→0.02로 비정상적으로 빨리 떨어진 걸 보면 모델이 "거의 다 Na"로
> 확신에 차서 찍는 쪽으로 편향 학습된 것으로 판단. Adaptive Thresholding(TH 클래스를 매 페어마다
> 학습)으로 교체해 이 문제를 학습 자체에서 구조적으로 처리하도록 수정 (`model.py`/`train_gat.py`
> 모듈 docstring에 진단 근거 기록). distant 단계엔 PUATLoss(na_weight=0.7, 기존 sweep으로 검증된
> 값), annotated 단계엔 자동으로 일반 ATLoss로 전환 — `Scripts/atlop/train_re.py`와 동일 패턴.
>
> Heterogeneous(Entity+Sentence) 그래프, Meta-path attention, Curriculum PU-weight 스케줄은
> 검토했으나 **의도적으로 보류** — 지금은 "손실함수 하나만 고쳐서 회복되는지"를 먼저 확인하고,
> 회복되면 그래프 구조 고도화를 별도 실험으로 진행 (변수 하나씩 바꾸는 원칙 유지).
>
> 이전: 제안 아키텍처(BERT + ATLOP LCP + 2-Layer Edge-featured GAT) 풀 학습 노트북 최초 작성.
> distant_epochs/annotated epochs/seed를 `Scripts/atlop` baseline과 동일하게 맞춰 아키텍처
> 변수만 비교하도록 수정. 파이프라인 상세는 `Scripts/dk_gat/README.md`.
>
> **Entity Pair Representation 명시**: GAT 노드만 단순 결합(`[v_i ‖ v_j]`, 1536-d)하는 게 아니라,
> ATLOP의 Localized Context Pooling 문맥벡터 `c_ij`(BERT 마지막 층 attention에서 head·tail이
> 공통으로 주목하는 토큰 분포로 추출한 768-d 벡터)를 반드시 함께 결합한다 —
> `z_ij = [v_i ‖ v_j ‖ c_ij] ∈ R^2304 → Linear → 768-d` (`model.py`의 `context`/`pair_proj`).
> `c_ij`는 GAT의 attention이 아니라 **인코더(BERT) 자체의 attention map**에서 나온다는 점이 핵심 —
> ATLOP 장점을 그대로 흡수한 구조.

**실행 전**: 런타임 → 런타임 유형 변경 → **A100 GPU**.

**학습 구성**: distant 20,000개 × **1 epoch** PUATLoss(na_weight=0.7) 프리트레인 → annotated
3,053개 × **15 epoch** ATLoss 파인튜닝 (AdamW 2e-5, wd 0.01, warmup 6%, clip 1.0, seed 66) —
**`Scripts/atlop` baseline과 distant_limit/distant_epochs/epochs/seed를 전부 동일하게 맞춰서
GAT 추가라는 아키텍처 변수 하나만 비교**함. 예측은 Adaptive Thresholding으로 결정(전역 threshold
없음, 페어마다 학습된 TH 클래스와 비교) — 더 이상 threshold sweep 불필요.

**비교 기준** (`results/comparison.md`, 전부 동일 스코어러·seed 66·distant 20k·distant_epochs 1):

| 모델 | annotated epochs | dev F1 | Ign F1 |
|---|---|---|---|
| ATLOP baseline | 15 | 61.71 | 59.86 |
| ATLOP + PUATLoss(0.7) | 15 | 62.06 | 60.16 |
| **dk EGAT (이 노트북)** | 15 | | |

In [1]:
# 0) GPU 확인 (A100이 보여야 함)
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-bfacc661-c2ed-6655-72b8-12c6f6a3ac8d)


In [ ]:
# 1) 코드 + 데이터 (Git LFS에서 필요한 json만 선별 pull)
#    Scripts/dk_gat는 아직 main이 아니라 dk 브랜치에만 있으므로 반드시 checkout 필요
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/multiful/Information_Extraction.git
%cd Information_Extraction
!git checkout dk
!git lfs pull --include="docred_data/data/dev.json,docred_data/data/train_annotated.json,docred_data/data/train_distant.json,docred_data/data/rel_info.json"
# json들이 MB~GB 단위로 보이면 성공 (133바이트면 LFS pull 실패)
!ls -lh docred_data/data/*.json
# Scripts/dk_gat가 실제로 있는지 확인 (여기 안 뜨면 아래 학습 셀에서 ModuleNotFoundError 남)
!ls Scripts/dk_gat/

In [3]:
# 2) 패키지
!pip install -q transformers==4.57.6 accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 131.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [9]:
# 3) 학습: distant 20k×1ep -> annotated 15ep (baseline과 동일 스케줄, A100 약 2~3시간)
#    epoch마다 [스테이지 | epoch N] dev_F1=.. Ign_F1=.. 로그가 찍힘
#    --use_pu_loss --na_weight 0.7: distant 단계만 PUATLoss, annotated는 자동으로 일반 ATLoss
#    (BCE+threshold sweep은 폐기 -- 실측 dev F1 24.77로 같은 20k 기준 RoBERTa+ATLoss의
#    43.15보다도 낮아서 Adaptive Thresholding으로 교체함, model.py 모듈 docstring 참고)
!python -m Scripts.dk_gat.train_gat \
  --distant_limit 20000 --distant_epochs 1 --epochs 15 \
  --use_pu_loss --na_weight 0.7 \
  --lr 2e-5 --weight_decay 0.01 --warmup_ratio 0.06 --dropout 0.1 \
  --run_name dk_gat --save_model --seed 66

usage: train_gat.py [-h] [--model_name_or_path MODEL_NAME_OR_PATH]
                    [--run_name RUN_NAME] [--epochs EPOCHS]
                    [--distant_epochs DISTANT_EPOCHS]
                    [--distant_limit DISTANT_LIMIT]
                    [--train_batch_size TRAIN_BATCH_SIZE]
                    [--distant_batch_size DISTANT_BATCH_SIZE]
                    [--eval_batch_size EVAL_BATCH_SIZE] [--lr LR]
                    [--weight_decay WEIGHT_DECAY]
                    [--warmup_ratio WARMUP_RATIO]
                    [--max_grad_norm MAX_GRAD_NORM] [--dropout DROPOUT]
                    [--gat_heads GAT_HEADS]
                    [--evidence_weight EVIDENCE_WEIGHT] [--seed SEED]
                    [--limit_docs LIMIT_DOCS] [--log_every LOG_EVERY]
                    [--save_model] [--device DEVICE]
train_gat.py: error: unrecognized arguments: --use_pu_loss --na_weight 0.7


In [ ]:
# 4) 결과 백업 (세션 끊기면 파일이 사라지므로 반드시 실행)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"
!cp results/dk_gat.pt results/dk_gat_stage1.pt results/dk_gat_dev_predictions.json \
   "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/"
!ls -lh "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/" | grep dk_gat

Mounted at /content/drive
cp: cannot stat 'results/dk_gat.pt': No such file or directory
cp: cannot stat 'results/dk_gat_stage1.pt': No such file or directory
cp: cannot stat 'results/dk_gat_dev_predictions.json': No such file or directory


## 결과 기록

마지막 epoch 로그 수치를 위의 비교표와 `results/comparison.md`에 반영.

- stage-1(distant 직후) 수치도 따로 적어두면 GAT가 노이즈 라벨 위에서 얼마나 버티는지,
  annotated 파인튜닝이 얼마나 끌어올리는지 분해해 보여줄 수 있음.
- seed 66 단일 시드 — baseline과의 차이가 ±1점 이내면 PRD §5 시드 노이즈 주의 적용.
- Evidence Contrastive Loss는 annotated 단계에서만 실질 작동함(distant는 evidence 없음).
- GAT 고도화 실험(edge feature ablation, 헤드 수, 인접행렬 반경 등)은 이 노트북의 셀 3에
  `--gat_heads`, `--evidence_weight` 인자만 바꿔 반복하면 됨.
- Heterogeneous(Entity+Sentence) 그래프/Meta-path/Curriculum PU-weight는 이번 회복 확인 후
  별도 실험으로 검토 예정 (지금 버전엔 없음).